# RUIP-BA: Valid claim and Invlid claim 
This program is primarily configured and tuned for the voice dataset (applying RP+Laplace noise), and most model parameters and hyperparameters are optimized for this setting. When applying the same framework to other datasets, these parameters generally need to be adjusted and re-tuned to ensure optimal performance.

The program processes oversampled training data from Group 1 by first projecting it into a lower-dimensional space using random projection. To ensure differential privacy, either Laplace or Gaussian noise is added to the projected features. The resulting privacy-preserved data is then used to train a machine learning model.

For the valid claim setting, the trained model is evaluated using test data from Group 1. The test data is transformed using the same random projection matrices and the same differential privacy configuration as used during training, ensuring full consistency between training and testing procedures.

For the invalid claim setting, the trained model is evaluated using test data from Group 2. In this case, a different random projection matrix is used for Group 2, introducing inconsistency in the feature transformation relative to the training pipeline. However, the differential privacy parameters (e.g., $\varepsilon$, $\delta$, and noise scale) remain unchanged, as they are treated as fixed and publicly known across both groups.

Overall, this program supports both valid and invalid claim settings across different datasets. It can operate on plain data as well as in configurations involving random projection and/or differential privacy noise (Laplace or Gaussian), with appropriate adjustment of model parameters and hyperparameters. However, due to dataset-specific tuning and sensitivity to these configurations, the numerical results may vary from those reported in this paper when different parameter or hyperparameter settings are used.

In [ ]:
# =====================================================
# Hyperparameters and Configuration for model training
# =====================================================

# Dataset paths
DATASET_PATH = 'Dataset/OversampledVoiceData.csv'
TESTDATASET_PATH = 'Dataset/VoiceDatatest.csv'

# Profile separation
SEPARATE_PROFILE = 68

# Random projection parameters
USE_RANDOM_PROJECTION = True

if USE_RANDOM_PROJECTION==False:
    N_COMPONENTS = 104
else:
   N_COMPONENTS = 94
   

# Differential privacy parameters
USE_LAPLACE_NOISE = True
USE_GAUSSIAN_NOISE = False
EPSILON = 7.0
SENSITIVITY = 1.0
DELTA = 1e-5

# Input dimension for neural network
INPUT_DIM = N_COMPONENTS
BATCH_SIZE = 32
EPOCHS = 50

# Dynamic column generation
column1 = [f'RPF{i+1}' for i in range(N_COMPONENTS)]

# =====================================================
# Load dataset
# =====================================================

import pandas as pd
import numpy as np

dataset = pd.read_csv(DATASET_PATH, index_col=0)

In [246]:
# Separate the profiles into two groups:
# (i) training profiles (0-SEPARATE_PROFILE-1)
# (ii) auxiliary profiles (SEPARATE_PROFILE and above)

totalUser = len(pd.unique(dataset['Label']))

trainingData = dataset[dataset['Label'] < SEPARATE_PROFILE]
auxilaryData = dataset[dataset['Label'] >= SEPARATE_PROFILE]

print("Total user in training dataset:", len(pd.unique(trainingData['Label'])))
print("Total user in auxiliary dataset:", len(pd.unique(auxilaryData['Label'])))

Total user in training dataset: 68
Total user in auxiliary dataset: 18


In [247]:
#Random Project
from sklearn.random_projection import SparseRandomProjection

def apply_random_projection(data, total_user, n_components=N_COMPONENTS):
    
    datasetRP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user):  
        rng = np.random.RandomState(seed)
        X = data[data['Label']==seed]
        transformer = SparseRandomProjection(n_components=N_COMPONENTS, random_state=rng)
        Xdata=X.drop(columns=['Label'])
        XRP = pd.DataFrame(transformer.fit_transform(Xdata),columns=column1)
        XRP['Label']=seed
        datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)    
    return datasetRP

In [248]:
# Laplace noise
def add_laplace_noise(data, total_user, epsilon=EPSILON, sensitivity=SENSITIVITY):
    datasetRPDP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user):
        X = data[data['Label']==seed]
        Xdata=X.drop(columns=['Label'])
        
        scale = sensitivity / epsilon
        noise = np.random.laplace(loc=0.0, scale=scale, size=Xdata.shape)
        Xdata=Xdata+noise 
        
        XdataRPDP = pd.DataFrame(Xdata,columns=column1)
        XdataRPDP['Label']=seed
        datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)
    return datasetRPDP

In [249]:
# Gaussioan noise
def add_gaussian_noise(data, total_user,epsilon=EPSILON,sensitivity=SENSITIVITY,delta=DELTA):
    datasetRPDP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user):
        X = data[data['Label']==seed]
        Xdata=X.drop(columns=['Label'])
        
        sigma = (sensitivity * np.sqrt(2 * np.log(1.25 / delta))) / epsilon
        noise = np.random.normal(0, sigma)
        Xdata=Xdata+noise 
        
        XdataRPDP = pd.DataFrame(Xdata,columns=column1)
        XdataRPDP['Label']=seed
        datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)
    return datasetRPDP

In [250]:
# =====================================================
# Feature Processing Pipeline
# =====================================================

feature_data = trainingData
total_user=len(pd.unique(trainingData['Label']))
# Apply Random Projection
if USE_RANDOM_PROJECTION:
    feature_data = apply_random_projection(feature_data, total_user)

# Apply Laplace Noise
if USE_LAPLACE_NOISE:
    feature_data = add_laplace_noise(feature_data, total_user)

# Apply Gaussian Noise
if USE_GAUSSIAN_NOISE:
    feature_data = add_gaussian_noise(feature_data, total_user)

# Convert processed features to DataFrame
feature_data = pd.DataFrame(feature_data)


# Final processed dataset
processed_dataset = feature_data

#print("Total user in the training dataset:", len(pd.unique(processed_dataset['Label'])))
#print(processed_dataset.head())

C:\Users\mdmor\AppData\Local\Temp\ipykernel_73860\1979874663.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)
C:\Users\mdmor\AppData\Local\Temp\ipykernel_73860\3624683841.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)


In [251]:
#Prepare the traning data for training and testing the model
import tensorflow
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

if USE_RANDOM_PROJECTION==False:
    X=trainingData.drop(columns=['Label'])
    y=trainingData['Label']
else:
    X=processed_dataset.drop(columns=['Label'])
    y=processed_dataset['Label']

Xtrain, Xval, ytrain, yval = train_test_split(X, y, test_size=0.2, random_state=22)

ytrain = to_categorical(ytrain)
yval = to_categorical(yval)
print(Xtrain.shape)
print(ytrain.shape)
print(Xval.shape)
print(yval.shape)

(10880, 94)
(10880, 68)
(2720, 94)
(2720, 68)


In [252]:
# import all necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#matplotlib inlineimport keras
from keras.layers import Dense, Dropout, Input,Activation,Dropout, Flatten
from keras.models import Model,Sequential
from keras.datasets import mnist

from keras.layers import BatchNormalization
from keras.optimizers import SGD, RMSprop, Adam
def adam_optimizer():
    return Adam(learning_rate=0.0002, beta_1=0.5)

def RMSprop_optimizer():
    return RMSprop(learning_rate=0.001, rho=0.9)

In [253]:
#neural network architecture

def create_classifierRP(release=False, totalClass=SEPARATE_PROFILE):
  classifier = Sequential()
  classifier.add(Dense(128, input_dim=N_COMPONENTS))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.5))

  #classifier.add(Dense(256))
  #classifier.add(BatchNormalization())
  #classifier.add(Activation('relu'))

  classifier.add(Dense(128))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.2))

  classifier.add(Dense(256))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.2))

  #classifier.add(Dense(256))
  #classifier.add(BatchNormalization())
  #classifier.add(Activation('relu'))

  classifier.add(Dense(128))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.2))

  #if release:
  classifier.add(Dense(totalClass, activation='softmax'))
  #else:
  #   classifier.add(Dense(Tuser))
  #np.log_softmax_v2(a, axis=axis)
  #classifier.add(F.softmax(a, dim=1))

  classifier.compile(loss='categorical_crossentropy', optimizer=RMSprop_optimizer(),metrics=['accuracy'])
  return classifier

Clasf=create_classifierRP()
Clasf.summary()

C:\Users\mdmor\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_27"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_148 (Dense)               │ (None, 128)            │        12,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_121         │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_121 (Activation)     │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_121 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_149 (Dense)               │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_122         │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_122 (Activation)     │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_122 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_150 (Dense)               │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_123         │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_123 (Activation)     │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_123 (Dropout)           │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_151 (Dense)               │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_124         │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_124 (Activation)     │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_124 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_152 (Dense)               │ (None, 68)             │         8,772 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105,924 (413.77 KB)

 Trainable params: 104,644 (408.77 KB)

 Non-trainable params: 1,280 (5.00 KB)

In [254]:
#Train the classifier
from keras.callbacks import ReduceLROnPlateau

learning_rate_reduction = ReduceLROnPlateau(monitor='val_acc', patience=5, verbose=1, factor=0.5,min_lr=0.0001)
callbacks_list = [learning_rate_reduction]

Classfier2= create_classifierRP(True,68)

#------Comment will start from here
lossc='categorical_crossentropy'
optimizerc=RMSprop(learning_rate=0.001, rho=0.9)
Classfier2.compile(loss=lossc, optimizer=optimizerc,metrics=['accuracy'])
#------Comments will end from here
historyc2 =  Classfier2.fit(Xtrain, ytrain, batch_size=64, epochs=200, validation_data=(Xval, yval),verbose=1)

Epoch 1/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.2265 - loss: 3.3933 - val_accuracy: 0.9261 - val_loss: 1.4987
Epoch 2/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8185 - loss: 0.9476 - val_accuracy: 0.9993 - val_loss: 0.0477
Epoch 3/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9176 - loss: 0.3703 - val_accuracy: 1.0000 - val_loss: 0.0025
Epoch 4/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9481 - loss: 0.2094 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 5/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9540 - loss: 0.1802 - val_accuracy: 1.0000 - val_loss: 1.7884e-04
Epoch 6/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9664 - loss: 0.1235 - val_accuracy: 1.0000 - val_loss: 1.0680e-04
Epoch 7/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9662 - loss: 0.1128 - val_accuracy: 1.0000 - val_loss: 5.8083e-05
Epoch 8/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9673 - loss: 0.10

In [262]:
# =====================================================
# Hyperparameters and Configuration for valid and invalid claim
# =====================================================
#Type of claim
VALID_CLAIM = False
INVALID_CLAIM =True

In [263]:
#Random Project for vlaid and invalid claim
from sklearn.random_projection import SparseRandomProjection

def apply_random_projectionClaim(data, total_user, n_components=N_COMPONENTS):
    
    datasetRP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user): 
        if VALID_CLAIM==True: 
            rng = np.random.RandomState(seed)
        else:
             rng = np.random.RandomState(seed+SEPARATE_PROFILE)
        X = data[data['Label']==seed]
        transformer = SparseRandomProjection(n_components=N_COMPONENTS, random_state=rng)
        Xdata=X.drop(columns=['Label'])
        XRP = pd.DataFrame(transformer.fit_transform(Xdata),columns=column1)
        XRP['Label']=seed
        datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)    
    return datasetRP

In [264]:
#read the test data either for valid or invalid claim
import csv
import pandas as pd

if VALID_CLAIM:
    testdataset = pd.read_csv(TESTDATASET_PATH, index_col=0)
    testdataset = testdataset[testdataset['Label'] < SEPARATE_PROFILE]
    
if INVALID_CLAIM:
    testdataset = pd.read_csv(TESTDATASET_PATH, index_col=0)
    testdataset = testdataset[testdataset['Label'] >= SEPARATE_PROFILE]
    newID = np.random.randint(0, SEPARATE_PROFILE, size=testdataset.shape[0])
    testdataset['Label'] = newID
    
testdataset.head()

,1,2,3,4,5,6,7,8,9,10,...,96,97,98,99,100,101,102,103,104,Label
8796,0.577301,-0.521846,-0.458939,-0.169801,0.492290,-0.608976,-0.075974,-0.057947,0.272287,-0.016334,...,0.048005,-0.225233,-0.009301,0.182901,-0.043218,-0.021152,0.009831,0.063303,-0.055561,2
8209,0.327843,-0.168374,0.086000,0.294810,-0.125812,-0.565006,0.021564,0.336860,-0.243190,-0.249655,...,-0.293992,-0.315545,0.182196,0.142055,-0.040365,-0.009444,-0.190533,0.016862,-0.503840,56
8721,0.436850,0.242402,-0.545839,0.490844,-0.176363,0.218081,-0.479460,0.485923,-0.089936,0.129415,...,0.454873,-0.007616,-0.550065,0.344938,-0.151026,0.131478,0.022711,-0.051380,0.116940,9
8875,0.824280,-0.238647,-0.423929,-0.034956,0.228715,-0.374185,-0.142289,0.114626,0.166814,0.080253,...,0.069756,-0.344372,-0.107741,0.179268,0.074635,-0.068413,-0.087623,0.231988,-0.060065,40
9294,0.537243,0.459554,0.087789,-0.018138,-0.026348,0.170799,-0.129531,-0.216600,-0.362393,0.169105,...,-0.104851,-0.217039,0.072538,0.095208,0.076355,0.045516,-0.025971,0.003157,-0.144510,14


In [265]:
# =====================================================
# Feature Processing Pipeline
# =====================================================

feature_data = testdataset
total_user=len(pd.unique(testdataset['Label']))


# Apply Random Projection
if USE_RANDOM_PROJECTION:
    feature_data = apply_random_projectionClaim(feature_data, total_user)

# Apply Laplace Noise
if USE_LAPLACE_NOISE:
    feature_data = add_laplace_noise(feature_data, total_user)

# Apply Gaussian Noise
if USE_GAUSSIAN_NOISE:
    feature_data = add_gaussian_noise(feature_data, total_user)

# Convert processed features to DataFrame
feature_data = pd.DataFrame(feature_data)

# Final processed dataset
processed_dataset = feature_data

#print("Total user in the training dataset:", len(pd.unique(processed_dataset['Label'])))
#print(processed_dataset.head())

C:\Users\mdmor\AppData\Local\Temp\ipykernel_73860\432162272.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)
C:\Users\mdmor\AppData\Local\Temp\ipykernel_73860\3624683841.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)


In [266]:
if USE_RANDOM_PROJECTION==False:
    Xtest=testdataset.drop(columns=['Label'])
    ytest=testdataset['Label']
else:
    Xtest=processed_dataset.drop(columns=['Label'])
    ytest=processed_dataset['Label']
    
ytest = to_categorical(ytest)

In [267]:
#Performance of the classifier
Classfier2.compile(loss='categorical_crossentropy', optimizer=Adam(),metrics=['accuracy'])
loss, accuracy = Classfier2.evaluate(Xtest, ytest)
print('Loss:', loss)
print('Accuracy:', accuracy)

14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.0069 - loss: 9.9509       
Loss: 10.212560653686523
Accuracy: 0.0070588234812021255
